# Lab: Parallel HN fetch with httpx + tenacity
Fetch 100 top HN stories. Measure throughput. Target: >= 20 req/s.

In [ ]:
!pip install -q httpx tenacity

In [ ]:
import asyncio, time, httpx
from tenacity import retry, stop_after_attempt, wait_exponential_jitter, retry_if_exception_type

HN = 'https://hacker-news.firebaseio.com/v0'

@retry(
    retry=retry_if_exception_type((httpx.TransportError, httpx.HTTPStatusError)),
    wait=wait_exponential_jitter(initial=0.5, max=8),
    stop=stop_after_attempt(5),
    reraise=True,
)
async def fetch(client, url):
    r = await client.get(url, timeout=10)
    if r.status_code in (429, 503):
        raise httpx.HTTPStatusError('backoff', request=r.request, response=r)
    r.raise_for_status()
    return r.json()

async def top_stories(n=100, concurrency=20):
    limits = httpx.Limits(max_connections=concurrency, max_keepalive_connections=10)
    async with httpx.AsyncClient(limits=limits) as client:
        ids = (await fetch(client, f'{HN}/topstories.json'))[:n]
        sem = asyncio.Semaphore(concurrency)
        async def bounded(i):
            async with sem:
                return await fetch(client, f'{HN}/item/{i}.json')
        return await asyncio.gather(*(bounded(i) for i in ids))

t0 = time.perf_counter()
stories = await top_stories(100)
elapsed = time.perf_counter() - t0
print(f'fetched={len(stories)} elapsed={elapsed:.2f}s rps={len(stories)/elapsed:.1f}')

No LLM call in this lab. cost = $0.00 / run.